### Lendo dados:
#### Vamos pegas os dados da olist_order_dataset

## Ideia de plano de ação:
### limpar os Nan
### limpar os status canceled
#### para métrica gerar um .csv:
##### tempo para aprovar o pedido.
##### tempo em transito.
##### diferença entre tempo de entrega_estimativa/realidade
##### tempo total da aprocação até a entrega:

In [1]:
import pandas as pd

In [2]:
data_path = '../data_raw/olist_orders_dataset.csv'

In [36]:
df = pd.read_csv(data_path)
len(df)

99441

In [16]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

### Passando por todas as colunas que não tem NaN e tirar no order statuds o canceled

In [35]:
df = pd.read_csv(data_path)
df_filtrado  = df[df['order_purchase_timestamp'].notna()]
df_filtrado = df_filtrado[df_filtrado['order_approved_at'].notna()]
df_filtrado = df_filtrado[df_filtrado['order_delivered_carrier_date'].notna()]
df_filtrado = df_filtrado[df_filtrado['order_delivered_customer_date'].notna()]
df_filtrado = df_filtrado[df_filtrado['order_estimated_delivery_date'].notna()]
df_filtrado = df_filtrado[df_filtrado['order_status'] != 'canceled']
len(df_filtrado)

96455

### Aqui vai calcular o tempo para aprovação:

In [44]:
df_time_to_approved= df_filtrado[['order_id', 'customer_id','order_purchase_timestamp','order_approved_at']].copy()
df_time_to_approved['order_purchase_timestamp'] = pd.to_datetime(df_time_to_approved['order_purchase_timestamp'])
df_time_to_approved['order_approved_at'] = pd.to_datetime(df_time_to_approved['order_approved_at'])
df_time_to_approved['time_to_aproved'] = df_time_to_approved['order_approved_at'] - df_time_to_approved['order_purchase_timestamp']
df_time_to_approved.head()

,order_id,customer_id,order_purchase_timestamp,order_approved_at,time_to_aproved
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-02 11:07:15,0 days 00:10:42
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-07-26 03:24:27,1 days 06:42:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-08 08:55:23,0 days 00:16:34
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-11-18 19:45:59,0 days 00:17:53
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-13 22:20:29,0 days 01:01:50


### Aqui vai calcular o tempo em transito:

In [ ]:
df_coparation_time = df_filtrado[['order_id', 'customer_id','order_delivered_carrier_date','order_delivered_customer_date']].copy()
df_coparation_time['order_delivered_customer_date'] = pd.to_datetime(df_coparation_time['order_delivered_customer_date'])
df_coparation_time['order_delivered_carrier_date'] = pd.to_datetime(df_coparation_time['order_delivered_carrier_date'])
df_coparation_time['transit_time'] = df_coparation_time['order_delivered_customer_date'] - df_coparation_time['order_delivered_carrier_date']
df_coparation_time.head()

,order_id,customer_id,order_delivered_carrier_date,order_delivered_customer_date,transit_time
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-04 19:55:00,2017-10-10 21:25:13,6 days 01:30:13
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-26 14:31:00,2018-08-07 15:27:45,12 days 00:56:45
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 13:50:00,2018-08-17 18:06:29,9 days 04:16:29
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-22 13:39:59,2017-12-02 00:28:42,9 days 10:48:43
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-14 19:46:34,2018-02-16 18:17:02,1 days 22:30:28


### Aqui vai calcular a diferença entre a estimativa e relalidade da entrega

In [51]:
df_coparation_time = df_filtrado[['order_id', 'customer_id','order_estimated_delivery_date','order_delivered_customer_date']].copy()
df_coparation_time['order_delivered_customer_date'] = pd.to_datetime(df_coparation_time['order_delivered_customer_date'])
df_coparation_time['order_estimated_date'] = pd.to_datetime(df_coparation_time['order_estimated_delivery_date'])
df_coparation_time['comparation_time'] = df_coparation_time['order_estimated_date'] - df_coparation_time['order_delivered_customer_date']
df_coparation_time.head()

,order_id,customer_id,order_estimated_delivery_date,order_delivered_customer_date,order_estimated_date,comparation_time
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-18 00:00:00,2017-10-10 21:25:13,2017-10-18,7 days 02:34:47
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-08-13 00:00:00,2018-08-07 15:27:45,2018-08-13,5 days 08:32:15
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-09-04 00:00:00,2018-08-17 18:06:29,2018-09-04,17 days 05:53:31
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-12-15 00:00:00,2017-12-02 00:28:42,2017-12-15,12 days 23:31:18
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-26 00:00:00,2018-02-16 18:17:02,2018-02-26,9 days 05:42:58


### Aqui vai calcular o tempo total, do início do pedido até a entrega:

In [54]:
df_total_time = df_filtrado[['order_id', 'customer_id','order_purchase_timestamp','order_delivered_customer_date']].copy()
df_total_time['order_delivered_customer_date'] = pd.to_datetime(df_total_time['order_delivered_customer_date'])
df_total_time['order_purchase_timestamp'] = pd.to_datetime(df_total_time['order_purchase_timestamp'])
df_total_time['total_time'] = df_total_time['order_delivered_customer_date'] - df_total_time['order_purchase_timestamp']
df_total_time.head()

,order_id,customer_id,order_purchase_timestamp,order_delivered_customer_date,total_time
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-10 21:25:13,8 days 10:28:40
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-08-07 15:27:45,13 days 18:46:08
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-17 18:06:29,9 days 09:27:40
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-12-02 00:28:42,13 days 05:00:36
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-16 18:17:02,2 days 20:58:23


In [56]:
data_path_filtred = '../data_processed/data_processed.csv'
df_final = pd.read_csv(data_path_filtred)